# CYR-GPU-014 / R1C — Fixed-Matrix Softmax Mechanism Dissection

This is a deliberately **large, multi-session, exact-resumable** experiment. Every arm has the same physical 24,576-row tied Cymek V5 model; only training-time output competition changes.

Select **T4 GPU**, then **Runtime → Run all**. If the session ends with `PARTIAL_SESSION`, keep the Drive folder and run this notebook again in another T4 session. The frozen protocol resumes; no seeds/arms/exposure are reduced.


In [ ]:
# CELL 0 — live readiness -> frozen checkout -> static tests -> CUDA preexecution gate
import os, sys, json, hashlib, subprocess
from pathlib import Path
REPO=Path('/content/An-Ra-the-new-AGI-r1c')
REMOTE='https://github.com/dhurv0045com-spec/An-Ra-the-new-AGI.git'
BRANCH='cymek-500m-readiness'
EXEC='2a71cea10ebb7a231834b6b112c49e268e9631a5'
PRE_REL='docs/cymek/experiments/CYR-GPU-014-R1C/PREREGISTRATION.json'
READY_REL='docs/cymek/experiments/CYR-GPU-014-R1C/RUN_READINESS.json'
if not REPO.exists():
    subprocess.run(['git','clone','--branch',BRANCH,'--single-branch','--depth','160',REMOTE,str(REPO)],check=True)
else:
    subprocess.run(['git','-C',str(REPO),'fetch','origin',BRANCH,'--depth','160'],check=True)
    subprocess.run(['git','-C',str(REPO),'checkout','-q',BRANCH],check=True)
    subprocess.run(['git','-C',str(REPO),'reset','--hard',f'origin/{BRANCH}'],check=True)
pre_text=(REPO/PRE_REL).read_text()
pre=json.loads(pre_text)
ready=json.loads((REPO/READY_REL).read_text())
assert pre['status']=='PREREGISTERED_BEFORE_SCIENTIFIC_EXECUTION'
assert pre['scientific_executable_commit']==EXEC
assert ready['status']=='READY_FOR_OPERATOR_COLAB_CUDA_PREEXECUTION_GATE'
assert ready['ready_for_operator_colab_gpu_run'] is True
assert ready['scientific_result_status']=='NOT_EXECUTED'
assert ready['frozen_scientific_executable_commit']==EXEC
PRE_LOCAL=Path('/content/CYR_GPU_014_R1C_PREREGISTRATION.json')
PRE_LOCAL.write_text(pre_text)
subprocess.run(['git','-C',str(REPO),'checkout','-q',EXEC],check=True)
head=subprocess.check_output(['git','-C',str(REPO),'rev-parse','HEAD'],text=True).strip(); assert head==EXEC
expected={
 'v5_experiments/cyr_gpu014_r1c.py':'1d7e76ebc4728ef9e4d21f46a40017cd5ffff327',
 'anra_v5/cyr_gpu014_r1c_run.py':'5bd880dea143e14348019b9c4f959c760a92f051',
 'tests/test_v5_cyr_gpu014_r1c.py':'998f45490c25763b1782992c728894153d9b4347',
 'docs/cymek/experiments/CYR-GPU-014-R1C/PLAN.md':'c45a24a50b5e6ef1ede9a380238fd45ad108bb2a',
 'v5_experiments/cyr_gpu012_r1.py':'d043c46ccf30c781f39c69d13c9597465767e188',
 'v5_experiments/cyr_gpu011.py':'600140d388ce94a6dc08fa32cf84d37511889362',
 'anra_v5/cyr_gpu011_run.py':'356523d9ee7fb0916c09c5806964892f9024dca7',
 'anra_v5/cyr_gpu006_run.py':'f61f17e9e2bdbefd8edc98dd7a7598a8dea35934',
 'v5_model/core.py':'7cf64b6f557a0556c074e5f61adfc86702f4c725',
 'v5_training/production_backend.py':'84d2dfc7cad3f52fd3a9b3f7357fac4d70145cb6',
 'v5_objectives/causal_lm.py':'4d0cc7c31679f99b44d34570e47e40f6778b7231',
 'docs/cymek/experiments/CYR-GPU-011/ARK002B_TASK_MANIFEST.json':'6c46fdf90139526b00e9041af2d511ed0ac24270'}
for p,sha in expected.items():
    got=subprocess.check_output(['git','-C',str(REPO),'hash-object',p],text=True).strip()
    assert got==sha,(p,got,sha)
subprocess.run([sys.executable,'-m','pip','install','-q','pytest','numpy'],check=True)
for p in ['v5_experiments/cyr_gpu014_r1c.py','anra_v5/cyr_gpu014_r1c_run.py']:
    subprocess.run([sys.executable,'-m','py_compile',str(REPO/p)],check=True)
subprocess.run([sys.executable,'-m','pytest','tests/test_v5_cyr_gpu014_r1c.py','tests/test_v5_cyr_gpu013_r1b.py','tests/test_v5_cyr_gpu012_r1.py','-q'],cwd=REPO,check=True)
import torch
if not torch.cuda.is_available(): raise RuntimeError('Select Runtime -> Change runtime type -> T4 GPU')
print('GPU:',torch.cuda.get_device_name(0),'VRAM GiB:',round(torch.cuda.get_device_properties(0).total_memory/2**30,2))
from google.colab import drive
drive.mount('/content/drive')
OUT=Path('/content/drive/MyDrive/CYMEK/CYR-GPU-014-R1C')
OUT.mkdir(parents=True,exist_ok=True)
binding={'scientific_executable_commit':EXEC,'preregistration_raw_sha256':hashlib.sha256(pre_text.encode()).hexdigest()}
binding_path=OUT/'EXECUTABLE_BINDING.json'
if binding_path.exists():
    old=json.loads(binding_path.read_text()); assert old==binding,('R1C output belongs to another frozen executable/preregistration',old,binding)
else:
    binding_path.write_text(json.dumps(binding,indent=2)+'\n')
gate_path=OUT/'PREEXECUTION_GATE.json'
campaign_path=OUT/'CAMPAIGN_RECEIPT.json'
reuse_gate=False
if gate_path.exists():
    g=json.loads(gate_path.read_text())
    if g.get('status')=='PASS': reuse_gate=True
if reuse_gate:
    print('Existing identity-bound R1C PREEXECUTION GATE: PASS — reusing for exact-resume session')
else:
    cmd=[sys.executable,'anra_v5/cyr_gpu014_r1c_run.py','--mode','preflight','--repo',str(REPO),'--out',str(OUT),'--prereg',str(PRE_LOCAL)]
    print('Running CUDA preflight/calibration:', ' '.join(cmd),flush=True)
    subprocess.run(cmd,cwd=REPO,check=True)
    g=json.loads(gate_path.read_text()); assert g.get('status')=='PASS'
resolved=json.loads((OUT/'RESOLVED.json').read_text())
print('R1C PREEXECUTION GATE: PASS')
print('Fixed protocol: 4 seeds x 6 arms x 3000 updates = 72,000 updates')
print('T4 conservative estimated total minutes:',round(resolved['estimated_total_campaign_seconds']/60,1))
print('Estimated 330-minute sessions:',resolved['estimated_sessions'])


In [ ]:
# CELL 1 — execute/resume one bounded scientific session
# Safe to run again after PARTIAL_SESSION. Completed arms are skipped and the current arm resumes from Drive.
import subprocess, sys
cmd=[sys.executable,'anra_v5/cyr_gpu014_r1c_run.py','--mode','run','--repo',str(REPO),'--out',str(OUT),'--prereg',str(PRE_LOCAL)]
print('Starting/resuming frozen R1C session...',flush=True)
subprocess.run(cmd,cwd=REPO,check=True)


In [ ]:
# CELL 2 — verify/download final or partial evidence bundle
import json, hashlib
from pathlib import Path
from google.colab import files
campaign=json.loads((OUT/'CAMPAIGN_RECEIPT.json').read_text())
status=campaign.get('status')
print('STATUS:',status)
print('COMPLETED ARMS:',campaign.get('completed_arm_count'),'/ 24')
print('VERDICT:',campaign.get('decision',{}).get('verdict'))
if status=='COMPLETE':
    bundle=OUT/'CYMEK_R1C_SOFTMAX_MECHANISM_RESULTS.zip'
else:
    bundle=OUT/'CYMEK_R1C_SOFTMAX_MECHANISM_PARTIAL.zip'
if not bundle.exists(): raise FileNotFoundError(bundle)
actual=hashlib.sha256(bundle.read_bytes()).hexdigest()
side=bundle.with_suffix(bundle.suffix+'.sha256')
if side.exists(): assert side.read_text().split()[0]==actual
print('BUNDLE:',bundle.name)
print('SHA256:',actual)
if status!='COMPLETE':
    print('R1C is intentionally multi-session. Keep the Drive folder. In the next T4 session Run all again (or rerun CELL 1 in this runtime). Do NOT change the protocol.')
else:
    print(json.dumps(campaign.get('decision',{}),indent=2)[:12000])
files.download(str(bundle))
